<a href="https://colab.research.google.com/github/jiyeonlee-2930/DeepLearning-TensorFlow-Basic/blob/main/GAN_Mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pylab as plt
import cv2
from tqdm.notebook import tqdm

# 파라미터 설정

In [ ]:
img_shape = (28, 28, 1)
z_dim = 100
row_num = 8
col_num = 8
batch_size = row_num * col_num
epoch_num = 10
learning_rate = 0.0001
class_num = 10

# MNIST데이터셋 불러오기

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_train = np.reshape(x_train, (-1, 28, 28, 1))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 저장할 디렉토리 생성
save_dir = "mnist_images"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 훈련 데이터에서 이미지 저장 (모든 이미지 또는 일부만 저장 가능)
num_images_to_save = 100  # 저장할 이미지 개수 (전체를 저장하려면 len(x_train)으로 설정)

for i in range(min(num_images_to_save, len(x_train))):
    # 0-1 범위의 이미지를 0-255 범위로 변환
    img = (x_train[i] * 255).astype(np.uint8)

    # 차원 변경 (28, 28, 1) -> (28, 28)
    img = img.reshape(28, 28)

    # PIL 이미지로 변환 후 저장
    img_pil = Image.fromarray(img)
    img_pil.save(os.path.join(save_dir, f"train_img_{i}_label_{y_train[i]}.png"))

    # 또는 matplotlib을 사용해 저장할 수도 있습니다
    # plt.figure(figsize=(1, 1))
    # plt.imshow(img, cmap='gray')
    # plt.axis('off')
    # plt.savefig(os.path.join(save_dir, f"train_img_{i}_label_{y_train[i]}.png"),
    #            bbox_inches='tight', pad_inches=0, dpi=100)
    # plt.close()

print(f"{num_images_to_save} 이미지가 {save_dir} 디렉토리에 저장되었습니다.")

# Discriminator(경찰) 모델 생성

In [ ]:
i=tf.keras.Input(shape=img_shape)
out = tf.keras.layers.Conv2D(16, 3, 2, padding='same')(i)
out = tf.keras.layers.Conv2D(32, 3, 2, padding='same')(out)
out = tf.keras.layers.Conv2D(64, 3, 2, padding='same')(out)
out = tf.keras.layers.Flatten()(out)
out = tf.keras.layers.BatchNormalization()(out)
out = tf.keras.layers.Dense(1024, activation='tanh')(out)
out = tf.keras.layers.Dense(1, activation='sigmoid')(out)
d_model = tf.keras.Model(inputs=[i],  outputs=[out])

d_model.summary()

# Generator(범죄자) 모델 생성

In [ ]:
i=tf.keras.Input(shape=(z_dim, ))
out = tf.keras.layers.Dense(1024, activation='tanh')(i)
out = tf.keras.layers.Dense(7*7*32, activation='tanh')(out)
out = tf.keras.layers.BatchNormalization()(out)
out = tf.keras.layers.Reshape((7, 7, 32))(out)
out = tf.keras.layers.Conv2DTranspose(16, 3, 2, padding='same')(out)
out = tf.keras.layers.Conv2DTranspose(1, 3, 2, padding='same')(out)
out = tf.keras.layers.Activation('sigmoid')(out)
g_model = tf.keras.Model(inputs=[i],  outputs=[out])

g_model.summary()

# GAN 학습

In [ ]:
# 옵티마이저
# 각 모델에 대한 별도의 옵티마이저 생성
d_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
g_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

# Discriminator 손실함수
d_loss = tf.keras.losses.BinaryCrossentropy()

In [ ]:
##생성 모델을 영상으로 저장
# 각 모델에 대한 별도의 옵티마이저 생성
d_opt = tf.keras.optimizers.Adam(learning_rate)
g_opt = tf.keras.optimizers.Adam(learning_rate)

# Discriminator 손실함수
d_loss = tf.keras.losses.BinaryCrossentropy()

fcc = cv2.VideoWriter_fourcc(*'DIVX')
out = cv2.VideoWriter('hjk_gan_mnist.avi', fcc, 1.0, (28*row_num, 28 * col_num))

# 1 Epoch에 batch size를 고려하여 학습할 수를 구한다.
batch_count = x_train.shape[0]//batch_size

# 비디오에 프레임을 추가하는 주기 설정 (모든 배치가 아닌 특정 간격으로)
video_sample_interval = 50  # 예: 50배치마다 이미지 추가

for e in range(epoch_num):
    for batch_idx in tqdm(range(batch_count)):

        # z는 Noise 또는 Latent Vector라 불리우는 값이다.
        z = np.random.uniform(-1.0, 1.0, (batch_size, z_dim))

        # predict 대신 직접 호출 사용
        with tf.GradientTape() as tape:
            f_img = g_model(z, training=False)  # training=False로 설정
            pred = d_model(f_img, training=True)
            f_label = np.zeros((batch_size, 1))
            loss = d_loss(f_label, pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        d_opt.apply_gradients(zip(grad, vars))

        # 실제 이미지 학습
        batch_num = np.random.randint(0, x_train.shape[0], size=batch_size)
        r_img = x_train[batch_num]
        r_label = np.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            pred = d_model(r_img, training=True)
            loss = d_loss(r_label, pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        d_opt.apply_gradients(zip(grad, vars))

        # Generator 학습
        z = np.random.uniform(-1.0, 1.0, (batch_size, z_dim))
        with tf.GradientTape() as tape:
            f_img = g_model(z, training=True)
            pred = d_model(f_img, training=False)  # discriminator는 학습하지 않음
            r_label = np.ones((batch_size, 1))
            loss = d_loss(r_label, pred)
        vars = g_model.trainable_variables
        grad = tape.gradient(loss, vars)
        g_opt.apply_gradients(zip(grad, vars))

        # 특정 간격으로만 이미지 샘플링 및 비디오 작성
        if batch_idx % video_sample_interval == 0:
            # 이미지 생성
            z_sample = np.random.uniform(-1.0, 1.0, (row_num * col_num, z_dim))
            sample_images = g_model(z_sample, training=False).numpy()

            # 효율적인 이미지 처리
            sample_images = sample_images.reshape(row_num, col_num, 28, 28)
            sample_img = np.block([[sample_images[i, j] for j in range(col_num)] for i in range(row_num)])

            sample_img = np.uint8(sample_img * 255.)
            sample_img = cv2.applyColorMap(sample_img, cv2.COLORMAP_HOT)
            out.write(sample_img)

    print(e, "완료")
out.release()

# Generator 가 생성하는 가짜 이미지가 원래 정답 레이블으 나타낼 수 있도록 하기 위해, One-Hot Encoding으로 레이블 값(l_i)을 Generator와 Discriminator의 입력으로 추가

In [ ]:
##Label을 입력받을 수 있도록 Discriminator수정
i=tf.keras.Input(shape=img_shape)
l_i = tf.keras.Input(shape=(1,), dtype=tf.int32)

# Fix: Wrap tf.one_hot in a Lambda layer with explicit output_shape
l_out = tf.keras.layers.Lambda(lambda x: tf.one_hot(x, class_num), output_shape=(1, class_num))(l_i)
l_out = tf.keras.layers.Dense(28*28*1)(l_out)
l_out = tf.keras.layers.Reshape((28, 28, 1))(l_out)

# Fix: Add labels to the image input
combined_input_d = tf.keras.layers.Add()([i, l_out])

# Fix: Start Conv2D layers with combined_input_d
out = tf.keras.layers.Conv2D(16, 3, 2, padding='same')(combined_input_d)
out = tf.keras.layers.Conv2D(32, 3, 2, padding='same')(out)
out = tf.keras.layers.Conv2D(64, 3, 2, padding='same')(out)
out = tf.keras.layers.Flatten()(out)
out = tf.keras.layers.BatchNormalization()(out)
out = tf.keras.layers.Dense(1024, activation='tanh')(out)
out = tf.keras.layers.Dense(1, activation='sigmoid')(out)
# Fix: Model inputs should include l_i
d_model = tf.keras.Model(inputs=[i, l_i], outputs=[out])

##Label을 입력값으로 받을 수 있도록 Generator모델 수정
i=tf.keras.Input(shape=(z_dim, ))
l_i = tf.keras.Input(shape=(1,), dtype=tf.int32)

# Fix: Wrap tf.one_hot in a Lambda layer with explicit output_shape
l_out = tf.keras.layers.Lambda(lambda x: tf.one_hot(x, class_num), output_shape=(1, class_num))(l_i)
l_out = tf.keras.layers.Dense(z_dim)(l_out)
l_out = tf.keras.layers.Reshape((z_dim,))(l_out)

# Fix: Combine noise and label output
combined_input_g = tf.keras.layers.Add()([i, l_out])

# Fix: Start Dense layers with combined_input_g
out = tf.keras.layers.Dense(1024, activation='tanh')(combined_input_g)
out = tf.keras.layers.Dense(7*7*32, activation='tanh')(out)
out = tf.keras.layers.BatchNormalization()(out)
out = tf.keras.layers.Reshape((7, 7, 32))(out)
out = tf.keras.layers.Conv2DTranspose(16, 3, 2, padding='same')(out)
out = tf.keras.layers.Conv2DTranspose(1, 3, 2, padding='same')(out)
out = tf.keras.layers.Activation('sigmoid')(out)
# Fix: Model inputs should include l_i
g_model = tf.keras.Model(inputs=[i, l_i], outputs=[out])

# Re-initialize optimizers and loss function after model redefinition
d_opt = tf.keras.optimizers.Adam(learning_rate)
g_opt = tf.keras.optimizers.Adam(learning_rate)
d_loss = tf.keras.losses.BinaryCrossentropy()

## Data를 생성하고 판단할 때 Label값을 받도록 수정
fcc = cv2.VideoWriter_fourcc(*'DIVX')
out_video = cv2.VideoWriter('cgan_mnist.avi', fcc, 1.0, (28*row_num, 28 * col_num)) # Rename 'out' to 'out_video' to avoid conflict with 'out' in model definitions

# 1 Epoch에 batch size를 고려하여 학습할 수를 구한다.
batch_count = x_train.shape[0]//batch_size

# 비디오에 프레임을 추가하는 주기 설정 (모든 배치가 아닌 특정 간격으로)
video_sample_interval = 50  # 예: 50배치마다 이미지 추가

for e in range(epoch_num):
    for batch_idx in tqdm(range(batch_count)):
        # f_y 값은 0~9 까지 임의의값을 원핫 인코딩한 값
        # f_y 값도 Generator 의 input값으로 추가

        # Discriminator 학습 (가짜 이미지)
        z = tf.random.uniform(minval=-1.0, maxval=1.0, shape=(batch_size, z_dim), dtype=tf.float32)
        f_y_disc = tf.convert_to_tensor(np.random.randint(0, class_num, size = batch_size).reshape((batch_size, 1)), dtype=tf.int32) # Labels for fake images in D training
        f_img = g_model([z, f_y_disc], training=False) # Fix: direct call with list input, training=False
        f_label = np.zeros((batch_size,1))

        with tf.GradientTape() as tape:
            pred = d_model([f_img, f_y_disc], training=True) # Fix: Pass both image and label to D
            loss = d_loss(f_label,pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        d_opt.apply_gradients(zip(grad, vars))

        # Discriminator 학습 (진짜 이미지)
        batch_num = np.random.randint(0, x_train.shape[0], size=batch_size)
        r_img = tf.convert_to_tensor(x_train[batch_num], dtype=tf.float32) # Convert r_img to TensorFlow tensor with explicit dtype
        r_y = tf.convert_to_tensor(y_train[batch_num].reshape((batch_size, 1)), dtype=tf.int32) # Fix: Get real labels, convert to tensor with explicit dtype
        r_label = np.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            pred = d_model([r_img, r_y], training=True) # Fix: Pass both image and label to D
            loss = d_loss(r_label, pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        d_opt.apply_gradients(zip(grad, vars))

        # Generator 학습
        z = tf.random.uniform(minval=-1.0, maxval=1.0, shape=(batch_size, z_dim), dtype=tf.float32) # Fix: Redefine z, convert to tensor
        f_y_gen = tf.convert_to_tensor(np.array([i%class_num for i in range(batch_size)]).reshape((batch_size,1)), dtype=tf.int32) # Fix: Labels for generator, convert to tensor with explicit dtype

        with tf.GradientTape() as tape:
            f_img = g_model([z, f_y_gen], training=True) # Fix: Pass noise and labels to G
            pred = d_model([f_img, f_y_gen], training=False)  # Fix: Pass fake image and its target label
            r_label = np.ones((batch_size, 1)) # Generator wants D to classify f_img as real
            loss = d_loss(r_label, pred)
        vars = g_model.trainable_variables
        grad = tape.gradient(loss, vars)
        g_opt.apply_gradients(zip(grad, vars))

        # 효율적인 이미지 처리
        if batch_idx % video_sample_interval == 0:
            # Generate images for visualization using fixed noise and labels
            z_sample = tf.random.uniform(minval=-1.0, maxval=1.0, shape=(row_num * col_num, z_dim), dtype=tf.float32)
            y_sample = tf.convert_to_tensor(np.array([i % class_num for i in range(row_num * col_num)]).reshape((row_num * col_num, 1)), dtype=tf.int32) # Generate corresponding labels, convert to tensor with explicit dtype
            gen_images = g_model([z_sample, y_sample], training=False).numpy()

            # Create a grid of images
            gen_images = gen_images.reshape(row_num, col_num, 28, 28)
            sample_img_grid = np.block([[gen_images[i, j] for j in range(col_num)] for i in range(row_num)])

            # Convert to uint8 and apply colormap
            sample_img_uint8 = np.uint8(sample_img_grid * 255.)
            color_img = cv2.applyColorMap(sample_img_uint8, cv2.COLORMAP_HOT)
            out_video.write(color_img) # Fix: Use out_video

    print(e, "완료")
out_video.release() # Fix: Use out_video

#